# Efficient ML: define the resource before optimizing it

**Core time:** 100 minutes. **Prerequisites:** neural layers and evaluation. **Outcomes:** calculate parameter storage and arithmetic counts; distinguish training/inference costs; identify activation and optimizer memory; define a multidimensional efficiency question.

## 1. “Smaller” and “faster” describe different quantities

A model can have fewer parameters and still run more slowly because its operators are poorly supported, it launches many small kernels or it moves data inefficiently. A compressed file can load into the same dense runtime representation. An unstructured sparse matrix can contain many zeros while a dense kernel still multiplies them. Therefore an efficiency claim must name its resource: parameters, serialized bytes, arithmetic, latency, throughput, peak memory, energy or training cost.

Quality also has several dimensions. A compressed model may preserve overall accuracy while degrading a rare class or calibration. The relevant optimization is usually constrained: minimize a resource while meeting a quality/latency/memory requirement, or expose the tradeoff without selecting a universal winner.

## 2. Count parameters and bytes carefully

A dense layer from d inputs to h outputs has `d*h+h` parameters with bias. A two-layer MLP with widths 64,32,10 has `64*32+32+32*10+10=2,410` parameters. If all are float32, raw parameter storage is 9,640 bytes. A serialized checkpoint has additional metadata; runtime memory includes more than parameters.



In [ ]:
d,h,c = 64,32,10
parameters = d*h+h+h*c+c
raw_parameter_bytes = parameters*4
assert parameters == 2410 and raw_parameter_bytes == 9640
print(parameters,raw_parameter_bytes)



During training, gradients and optimizer state add storage. Adam commonly keeps first and second moment tensors in addition to parameters, while activations saved for backpropagation can dominate memory. Mixed-precision training may retain master weights or other state depending on the implementation. Avoid a universal “training uses exactly four times the parameter memory” formula without defining the optimizer and precision policy.

## 3. Arithmetic counts need a convention

A dense matrix-vector product with d inputs and h outputs performs roughly d*h multiply-accumulate pairs per example, plus bias/activation work. Some reports count one multiply-add as two floating-point operations; others count one MAC. State the convention. For convolution, multiply the per-output channel/filter cost by the number of output spatial positions and batch size.

These counts are useful for reasoning but do not fully predict wall time. Memory bandwidth, caches, vectorization, parallelism, kernel selection and launch overhead matter. A workload can be compute-bound or memory-bound depending on its operational intensity and hardware. An optimized fused operator can outperform a theoretically smaller sequence of unfused operations.

## 4. Worked example: rank reduction may not reduce parameters

A matrix with shape `(m,n)` can be represented by factors `(m,r)` and `(r,n)`, requiring `r*(m+n)` values. This is fewer than `m*n` only when r is below `m*n/(m+n)`. For a 32-by-64 matrix, rank 8 uses 768 factor values instead of 2,048. Rank 24 uses 2,304, more than the dense matrix. A method named “low rank” does not automatically save storage at every chosen rank.

The factorized operator also performs two matrix products and creates an intermediate. Even when parameter count decreases, latency may increase for a small batch. That is why the laboratory measures rather than infers speed.

## 5. Design an efficiency contract

State the device, input shape, batch size, dtype, operator/backend, quality metric, allowable quality loss and resource target. For example: “On one CPU thread, batch-one digit inference, compare median and p95 latency while reporting test accuracy and raw/serialized parameter storage.” This is a narrow, measurable contract. It does not claim performance on a phone, GPU or production service.

Training-efficient methods may reduce trainable parameters without reducing inference parameters. LoRA is an example of a method whose adaptation cost and deployable representation need separate accounting. Distillation adds teacher-training cost even when the final student is smaller. Include the cost stage relevant to your claim.

## Exercises and self-check

1. Count parameters of a 64–16–10 MLP with bias. **Check:** 1,210.
2. Explain why zeroing half the weights does not halve dense parameter bytes. **Check:** the same tensors and dtypes remain allocated unless representation changes.
3. For a 100-by-100 matrix, determine the ranks that reduce factor storage. **Check:** integer r below 50; r=50 ties before metadata/bias.
4. Write separate contracts for training memory and batch-one inference latency. Identify measurements that one contract cannot support.

## Primary reading

- [Han, Mao and Dally: Deep Compression](https://arxiv.org/abs/1510.00149): distinguish pruning, quantization and coding stages rather than attributing all savings to zeros.
- [PyTorch: benchmark utilities](https://docs.pytorch.org/docs/2.9/benchmark_utils.html): connect operation-level measurement to a precisely specified workload.


# Measure latency, memory and throughput honestly

**Core time:** 120 minutes. **Prerequisites:** resource accounting and evaluation mode. **Outcomes:** design a repeatable benchmark; report distributions and units; separate batch latency from throughput; distinguish parameter bytes from measured peak memory.

## 1. Define the timed region

Decide whether timing includes preprocessing, host-to-device transfer, model execution and postprocessing. A model-only benchmark is useful, but it is not end-to-end service latency. Construct inputs and load weights outside the timed region unless those operations are intentionally part of the question. Put the model in evaluation mode and use an appropriate inference context.

Warm-up calls can initialize kernels, caches or compilation. Record how many were used and whether cold-start latency is reported separately. Repetitions produce a distribution; a single fastest call is not a representative latency estimate. Background processes, CPU frequency changes and thread settings can influence measurements.

## 2. Worked timing procedure

The CPU lab fixes PyTorch to one thread, performs 20 warm-up calls and records 100 individual calls with a high-resolution monotonic timer. It reports median and p95 batch latency in milliseconds for batch sizes 1 and 64. These are measurements on the executing machine, not predictions for a production deployment.



In [ ]:
import time
import numpy as np
import torch
torch.set_num_threads(1)
model = torch.nn.Linear(64,10).eval()
x = torch.zeros(1,64)
samples = []
with torch.inference_mode():
    for _ in range(20): model(x)
    for _ in range(100):
        start = time.perf_counter_ns()
        model(x)
        samples.append((time.perf_counter_ns()-start)/1e6)
print({'median_ms':float(np.median(samples)),
       'p95_ms':float(np.quantile(samples,.95))})



For extremely small operations, timer overhead and Python dispatch can be significant. A library benchmark timer can amortize repetitions and provide a more robust operator-level measurement. State the implementation used. On asynchronous accelerators, synchronize the relevant work or use appropriate device events; otherwise the host timer may measure enqueue time rather than completed computation.

## 3. Batch latency and throughput answer different questions

If a batch of 64 takes 4 ms, batch throughput is 16,000 examples per second under that measurement. This does not mean a single request has 0.0625 ms latency; it may wait for batch formation and share a larger execution. Report batch size beside every number. A system optimized for high throughput can be unsuitable for an interactive latency target.



In [ ]:
batch_size,batch_ms = 64,4.0
throughput = batch_size/(batch_ms/1000)
assert throughput == 16000
print(throughput)



Theoretical throughput derived from one batch measurement excludes queueing, input delivery and service overhead. Label it accordingly.

## 4. Memory measurements need names

Raw parameter bytes sum tensor elements times element size. Serialized file bytes come from the artifact on disk. Peak process memory includes Python/runtime/library allocations and may retain allocator caches. Device allocated/reserved memory are different counters. Activation memory depends on batch size and model execution; training also retains backward state.

The core lab reports raw parameter bytes and actual checkpoint/container sizes. It does not claim to measure peak process/device memory or energy. An extension can add platform-appropriate peak-memory instrumentation, but must define the measurement window and baseline. Do not label a parameter-count calculation “RAM usage.”

## 5. A fair benchmark compares the same workload

Use identical inputs, batch size, dtype, thread policy, warm-up and timed region across methods. Randomize or alternate method order to reduce drift effects when necessary. Repeat complete benchmark sessions if results are close. Report dispersion and avoid claiming a speedup from differences smaller than observed noise. Preserve raw timings so a reviewer can inspect outliers rather than accepting a handpicked average.

Energy needs an actual power/energy measurement or a clearly labeled model with assumptions. Fewer FLOPs do not directly establish fewer joules. Carbon estimates additionally depend on hardware utilization, energy mix and boundaries; this course does not infer them from parameter counts.

## Exercises and self-check

1. Why can GPU timing without synchronization be misleading? **Check:** operations may still be running when the host timer stops.
2. A pruned dense model is 2% faster in one run and 3% slower in another. Write a justified conclusion. **Check:** no clear speed benefit has been established under the current measurement variability.
3. Design cold-start and steady-state benchmarks and explain what each includes.
4. Identify the incorrect label in “model RAM=number of weights times four.” **Check:** this computes raw float32 parameter storage, not total runtime memory.

## Primary reading

- [PyTorch: benchmark utilities](https://docs.pytorch.org/docs/2.9/benchmark_utils.html): read warm-up, synchronization and measurement guidance.
- [Python: performance counter](https://docs.python.org/3/library/time.html#time.perf_counter): verify the timer's monotonic duration semantics.


# Pruning: sparsity, structure and the cost of zeros

**Core time:** 120 minutes. **Prerequisites:** neural layers and efficiency measurement. **Outcomes:** implement magnitude pruning; distinguish masked dense weights from sparse storage; explain structured removal; compare quality and cost with an appropriate baseline.

## 1. Specify what is removed

Unstructured pruning sets individual weights to zero. Structured pruning removes larger units such as channels, neurons or blocks. A mask multiplied into a dense tensor leaves its shape unchanged. A genuinely smaller structured network changes dimensions and downstream connections. These representations can have very different hardware behavior even when both report “50% sparsity.”

Magnitude pruning selects weights with small absolute values under a chosen scope: globally across selected layers or separately within each layer. Biases and normalization parameters may be excluded. State the scope and tie rule, because repeated magnitudes can otherwise change the exact sparsity count.

## 2. Worked example: exact selection

For weights `[-0.1,2.0,0.05,-3.0]`, pruning the two smallest magnitudes removes -0.1 and 0.05, leaving `[0,2,0,-3]`. The representation still contains four values if stored as a dense array. The Euclidean parameter change is `sqrt(0.1²+0.05²)`, but prediction change depends on inputs and network sensitivity.



In [ ]:
import numpy as np
w = np.array([-.1,2.,.05,-3.])
pruned = w.copy()
indices = np.argsort(np.abs(w),kind='stable')[:2]
pruned[indices] = 0
np.testing.assert_allclose(pruned,[0.,2.,0.,-3.])
assert pruned.nbytes == w.nbytes
print(pruned,np.linalg.norm(pruned-w))



A threshold rule can prune more than the requested count when many values tie at the threshold. Selecting exact sorted indices makes the intended count explicit. In larger models, the implementation should preserve a mask if pruning is meant to persist during fine-tuning; otherwise zeroed weights can regrow.

## 3. Prune, evaluate and optionally fine-tune

First save a trained baseline. Apply a fixed pruning rule to a copy, evaluate validation quality and record sparsity. If fine-tuning, use training data and an explicit budget; select the resulting checkpoint on validation. Compare with a baseline given an equivalent additional training opportunity if making a claim about the pruning method rather than merely the final artifact.

The core efficiency lab performs one-shot magnitude zeroing without fine-tuning. It therefore measures an immediate perturbation of a trained model, not the full iterative prune/retrain pipeline in Deep Compression. This scope is intentional and should appear in the report.

## 4. Sparse arithmetic requires support

A dense matrix multiplication generally still processes a zero-valued matrix entry according to its dense kernel. Sparse formats store values and locations and can save space at high enough sparsity, but metadata and irregular memory access add costs. Hardware-supported sparsity patterns can help, yet their constraints and kernels must match the representation. Do not infer speed from a zero counter.

Structured pruning can reduce dense dimensions, but removing a hidden unit requires removing its incoming weights/bias and corresponding outgoing connections. Batch normalization, residual connections and grouped convolutions can introduce additional consistency constraints. A model that executes after dimension surgery still needs accuracy and shape checks.

## 5. Compare the whole tradeoff

Report fraction of selected weights zeroed, overall parameter count, raw/serialized bytes, predictive quality and measured latency under the same workload. If file compression shrinks a zero-filled checkpoint, distinguish that disk benefit from in-memory storage. A slower result is plausible for small workloads and should not be suppressed.

## Exercises and self-check

1. Why can a masked dense tensor use more storage during training than the unpruned tensor? **Check:** the mask and possibly original parameter representation add state.
2. Remove one hidden neuron from a two-layer MLP and list all affected arrays. **Check:** first-layer row and bias, second-layer input column, plus any associated state.
3. Compare global and per-layer pruning at equal overall sparsity. Identify a case where one policy damages a small sensitive layer disproportionately.
4. Explain why a 50% sparse dense model cannot honestly be reported as 50% fewer allocated parameters.

## Primary reading

- [PyTorch: pruning tutorial](https://docs.pytorch.org/tutorials/intermediate/pruning_tutorial.html): inspect masks, reparameterization and removing pruning hooks.
- [Han et al.: Deep Compression](https://arxiv.org/abs/1510.00149): distinguish the paper's full pipeline from this one-shot teaching intervention.


# Quantization: representation error and actual execution

**Core time:** 120 minutes. **Prerequisites:** numerical precision and resource measurement. **Outcomes:** implement a simple quantizer; calculate clipping/rounding error; distinguish weight storage from integer inference; explain calibration and quantization-aware training.

## 1. A quantizer maps a continuous range to a finite codebook

Uniform affine quantization represents a real value approximately as a scale times an integer offset from a zero point. The code is rounded and clipped to the allowed integer range. Scale sets the spacing between representable values; clipping bounds the range. Both create error, and a few outliers can force a scale that is too coarse for most values.

```math
q=\operatorname{clip}(\operatorname{round}(x/s)+z,q_{min},q_{max}),
\qquad \hat x=s(q-z).
```

Symmetric quantization often uses zero point zero and a signed range; asymmetric quantization can represent an offset range. Per-tensor scaling shares one scale across a tensor, while per-channel scaling uses separate scales. Specify the axis and metadata format. “INT8” alone does not describe all these decisions.

## 2. Worked example: a symmetric weight quantizer

Use signed codes from -127 to 127 and choose `scale=max(abs(weight))/127`. For weights `[-1,0,0.5,1]`, the endpoints are represented exactly and the middle positive value is rounded to a neighboring code. Without clipping, nearest rounding produces at most half a scale of reconstruction error. A zero tensor needs an explicit nonzero scale convention to avoid division by zero.



In [ ]:
import numpy as np
w = np.array([-1.,0.,.5,1.],dtype=np.float32)
scale = max(float(np.abs(w).max())/127,np.finfo(np.float32).tiny)
q = np.clip(np.rint(w/scale),-127,127).astype(np.int8)
restored = q.astype(np.float32)*scale
assert np.max(np.abs(restored-w)) <= scale/2+1e-7
print(q,scale,restored)



The integer codes occupy one byte each, but the scale and any zero point also need storage. A file/container adds metadata. If restored weights are used by float32 matrix multiplication, inference is still floating-point execution; only the storage representation and parameter approximation were changed.

## 3. Weight-only simulation is not an integer kernel benchmark

The core lab quantizes weight tensors, saves integer codes and scales, then dequantizes to float32 for evaluation. It reports the packed artifact size and approximation effect separately from the float runtime model. Its latency measurement is explicitly labeled dequantized float inference. This prevents a common false claim: observing a smaller file and calling it faster INT8 inference without using an integer backend.

Actual integer inference must handle activation scales, accumulator precision, requantization, bias representation and supported operators. Backend/compiler support determines whether operations stay quantized or repeatedly convert. A conversion-heavy graph can lose the expected benefit. Validate the actual execution path before making a speed or energy claim.

## 4. Calibration and quantization-aware training

Post-training activation quantization often estimates ranges from representative calibration data. Those data belong to development, not the final test set. Poorly representative calibration can produce clipping on future inputs. Range-estimation choices and outlier handling are hyperparameters that should be recorded.

Quantization-aware training simulates quantization effects during optimization, often using a surrogate gradient for the nondifferentiable rounding operation. That surrogate is an optimization device, not the true derivative of rounding almost everywhere. The resulting artifact still needs conversion and backend validation. It should be compared with a baseline under a stated training budget.

## 5. Error should be measured at multiple levels

Measure weight reconstruction error, changes in logits and predictive quality. Small average weight error can still alter a decision near a boundary. Check classwise performance and calibration when relevant. Report saturation/clipping rates for activations if using them. A model with acceptable average accuracy can fail badly on rare but important input ranges.

## Exercises and self-check

1. Quantize an all-zero tensor and explain your scale convention. **Check:** avoid division by zero while restoring exact zeros.
2. If scale is 0.02 and no clipping occurs, what is the usual nearest-rounding absolute error bound? **Check:** 0.01, apart from finite-precision implementation tolerance.
3. Compare per-tensor and per-row scales on a matrix with one outlier row. Count scale metadata and reconstruction error.
4. Explain why the lab's packed artifact size and float inference latency cannot be combined into an “INT8 speedup” claim.

## Primary reading

- [Jacob et al.: Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference](https://arxiv.org/abs/1712.05877): inspect scale/zero-point arithmetic and the difference between training simulation and integer execution.
- [Han et al.: Deep Compression](https://arxiv.org/abs/1510.00149): compare quantized representations with pruning and coding stages.


# Low-rank factorization and LoRA: two different efficiency questions

**Core time:** 120 minutes. **Prerequisites:** SVD, dense layers and parameter accounting. **Outcomes:** replace a dense matrix with rank-r factors; verify exact reconstruction at full rank; distinguish compressing weights from adapting a frozen model; account for merged and unmerged inference.

## 1. Factor a matrix, then inspect the operator

For a weight matrix W with shape `(out,in)`, truncated SVD yields a rank-r approximation. Factor it into B `(out,r)` and A `(r,in)`. A dense layer `x@W.T+b` becomes `(x@A.T)@B.T+b`. The first factor has no bias; the original bias belongs after the second factor. Accidentally introducing two biases changes the represented function.

```math
W_r=U_r\Sigma_rV_r^\top=BA,\qquad
B=U_r\Sigma_r^{1/2},\quad A=\Sigma_r^{1/2}V_r^\top.
```

Splitting square-root singular values between the factors is convenient; placing all singular values in one factor is algebraically possible too. At full available rank, the reconstructed matrix should match the original within numerical tolerance. At lower rank, compare reconstruction error and task quality separately.

## 2. Worked example: storage and error

A 32-by-64 matrix contains 2,048 values. Rank 8 factors contain `32*8+8*64=768`, before bias. The Frobenius squared reconstruction error equals the sum of squared discarded singular values. If those singular values are small, matrix reconstruction is good, but classification can still change near a decision boundary. The SVD objective does not directly optimize prediction accuracy.



In [ ]:
import numpy as np
rng = np.random.default_rng(7)
W = rng.normal(size=(6,4))
U,s,Vt = np.linalg.svd(W,full_matrices=False)
r = 2
B = U[:,:r]*np.sqrt(s[:r])
A = np.sqrt(s[:r,None])*Vt[:r]
error = np.linalg.norm(W-B@A,'fro')**2
np.testing.assert_allclose(error,(s[r:]**2).sum())
np.testing.assert_allclose((U*s)@Vt,W,atol=1e-12)
print(error,B.size+A.size,W.size)



For this 6-by-4 example, rank 2 uses 20 values versus 24. A larger rank could fail to save storage. Report the actual dimensions and parameter counts, not only the chosen rank.

## 3. LoRA adapts a frozen base rather than discarding it

LoRA parameterizes a trainable update to a frozen weight matrix as a low-rank product, often scaled by alpha/r. The effective weight is the original W plus the update. Training fewer parameters can reduce some adaptation memory and optimizer state, but the base weights remain present. It is not equivalent to replacing W with its truncated SVD.

```math
W_{effective}=W_0+\frac{\alpha}{r}BA.
```

For a 1,000-by-1,000 base and rank 8, the adapter has 16,000 parameters, but the million base parameters still exist. Unmerged inference may compute both base and adapter paths. For suitable linear layers and a fixed adapter, merging the update into the base can remove the extra branch, while producing a full-sized effective weight. Discuss trainable count, total count and deployment representation separately.

## 4. Initialization and gradients matter

If both adapter factors are initialized to zero, gradients through their product can initially be zero for both. A common construction initializes one factor nonzero and the other zero so the initial update is zero while a gradient can enter one factor. The exact convention and scaling are implementation choices to record. Bias training and which layers receive adapters also change trainable counts.

The core lab demonstrates SVD factorization and a small LoRA arithmetic/gradient check. It does not fine-tune a large language model or claim the original paper's performance results. A full LoRA extension needs a task, frozen base, training protocol and appropriate memory/runtime measurements.

## Exercises and self-check

1. For a 512-by-512 matrix and rank 16, count adapter parameters and total base-plus-adapter parameters. **Check:** 16,384 and 278,528, before any biases.
2. Why can two smaller matrix products be slower than one larger one at batch one? **Check:** dispatch, intermediate storage and kernel efficiency can dominate arithmetic savings.
3. Verify full-rank factorization preserves logits, then vary rank using validation and keep test reserved.
4. Explain the difference between “98% fewer trainable parameters” and “98% smaller deployed model.” **Check:** frozen base storage and merged representation remain relevant.

## Primary reading

- [Hu et al.: LoRA](https://arxiv.org/abs/2106.09685): inspect the frozen-base update parameterization and adaptation claims.
- [MIT: Singular Value Decomposition](https://ocw.mit.edu/courses/18-06sc-linear-algebra-fall-2011/pages/positive-definite-matrices-and-applications/singular-value-decomposition/): connect matrix approximation to the factors implemented here.


# Knowledge distillation: a teacher, a student and a fair comparison

**Core time:** 2 hours. **Prerequisites:** logits, cross-entropy and training loops. **Outcomes:** define a temperature-scaled distillation objective; keep the teacher frozen; compare with an equally trained hard-label student; report teacher-training cost separately.

## 1. The student learns from more than the winning class

A teacher's probability distribution can encode relative preferences among incorrect classes. A student can learn from those soft targets alongside observed labels. This can be useful when the student has fewer parameters or a different architecture, but the teacher can also transmit errors, biases or overconfidence. Distillation is a training method, not a guarantee that the student becomes both smaller and equally accurate.

Use the same legal training inputs for the hard-label and distilled students unless an additional-data regime is explicitly part of the experiment. Teacher logits on test inputs must not become training targets. A teacher selected using test labels would also contaminate the comparison.

## 2. Temperature changes the target distribution

Dividing logits by a positive temperature greater than one softens the probability distribution. The student loss can combine ordinary hard-label cross-entropy with KL divergence from teacher probabilities to student probabilities at that temperature. A conventional T-squared factor compensates for the gradient scale introduced by temperature in this formulation.

```math
L=(1-\lambda)\,CE(y,z_s)+\lambda T^2\,KL\left(p_t(T)\;\|\;p_s(T)\right).
```

State which coefficient multiplies which term; papers and code can use opposite naming conventions. KL direction also matters. In PyTorch's common `kl_div(log_student,teacher_probability)` usage, the arguments are not both raw probabilities and the student input is log probability. Read the API rather than guessing from the function name.

## 3. Worked objective calculation

Suppose hard-label loss is 0.8, KL at temperature 2 is 0.1 and lambda is 0.5. The combined loss is `0.5*0.8 + 0.5*4*0.1 = 0.6`. Omitting T-squared would give 0.45, changing the relative gradient contribution. This arithmetic does not establish the best lambda or temperature; select those on validation under a stated budget.



In [ ]:
import torch
import torch.nn.functional as F
student = torch.tensor([[1.,0.,-1.]],requires_grad=True)
teacher = torch.tensor([[2.,1.,0.]])
y = torch.tensor([0])
T,weight = 2.,.5
soft = F.kl_div(F.log_softmax(student/T,dim=-1),
               F.softmax(teacher/T,dim=-1),reduction='batchmean')*T*T
loss = (1-weight)*F.cross_entropy(student,y)+weight*soft
loss.backward()
assert torch.isfinite(student.grad).all()
assert teacher.grad is None
print(float(loss.detach()))



In this particular example, the two logit vectors differ by a constant, so their softened probabilities coincide and KL is approximately zero. This is a useful invariance check. Change one teacher coordinate to make the soft term nonzero and inspect the resulting gradient.

## 4. Freeze the teacher and preserve the baseline

Set the teacher to evaluation mode and compute its targets without parameter gradients. Train one student on hard labels only and another with the combined objective, matching architecture, initialization seed, optimizer, epochs and selection rule. The distillation method adds teacher forward computation, so the training compute is not identical even when student update counts match. Report that difference.

The core efficiency lab trains a width-64 teacher and width-16 students. It uses fixed illustrative temperature and mixing weight, selects checkpoints on validation and evaluates test after selection. It does not assume the distilled student wins. If it does not, inspect teacher quality, student capacity, optimization and the chosen hyperparameters before generalizing.

## 5. Account for the lifecycle

A small student may reduce deployment cost while distillation increases development cost. If the teacher already exists, amortization differs from training a new teacher solely for one small deployment. Report teacher parameters, student parameters, training procedure and inference benchmark separately. A student can mimic unwanted teacher behavior, so rerun relevant robustness, calibration and group evaluations rather than assuming compression preserves all properties.

## Exercises and self-check

1. Explain why adding a constant to every teacher logit leaves soft targets unchanged. **Check:** softmax's numerator and denominator share the same exponential factor.
2. What comparison is missing if only teacher and distilled student are reported? **Check:** the same student trained without distillation under a comparable selection/update budget.
3. Verify that teacher weights do not change during student training.
4. Design a temperature sweep using validation and state how you will avoid test-driven selection.

## Primary reading

- [Hinton, Vinyals and Dean: Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531): inspect temperature and the motivation for soft targets.
- [PyTorch: KLDivLoss](https://docs.pytorch.org/docs/2.9/generated/torch.nn.KLDivLoss.html): verify input/log-target and reduction conventions.


# Efficiency laboratory: build and interpret a Pareto frontier

**Core time:** 3–4 hours. **Prerequisites:** Efficient ML 1–6. **Outcomes:** compare several representations under one workload; distinguish measured and analytical metrics; identify nondominated options; make a bounded deployment recommendation.

## 1. The laboratory compares mechanisms, not slogans

Run `python lab06_efficiency.py --output outputs/efficiency`. The experiment trains a reference MLP, applies one-shot magnitude pruning, evaluates an INT8 storage/dequantization approximation, replaces a layer with SVD factors and compares small hard-label/distilled students. Every method has a distinct scope: pruning remains dense, quantization is dequantized float execution, SVD changes layer structure and distillation changes training.

Read the method labels before comparing numbers. A compressed container size and a dense inference latency measure different representations. The report should make these distinctions visible rather than collecting them under one ambiguous “model size” column.

## 2. Pareto dominance is a precise comparison

For a quality metric to maximize and a cost to minimize, method A dominates B if A is at least as good in both and strictly better in at least one. Nondominated methods form a Pareto set for the measured points. The set depends on the chosen metrics, workload and uncertainty. It is not a proof that no untested method can do better.



In [ ]:
points = [('A',.90,10),('B',.92,12),('C',.89,13),('D',.91,9)]
frontier = []
for name,quality,cost in points:
    dominated = any(q>=quality and c<=cost and (q>quality or c<cost)
                    for other,q,c in points if other!=name)
    if not dominated: frontier.append(name)
assert frontier == ['B','D']
print(frontier)



These are illustrative quality/cost pairs, not measured laboratory results. A is dominated by D; C is also dominated. B and D trade higher quality against lower cost. A decision requires a constraint or preference, such as maximum latency or minimum acceptable quality.

## 3. Benchmark protocol

Use the saved split, fixed validation-selection rules and identical CPU/thread/input policies. Record raw timings for batch one and batch 64, warm-up count, repetitions, median/p95 and environment. Compare raw parameter bytes, actual serialized artifacts and predictive quality. Do not label these as peak RAM or energy. If adding those measurements, define an appropriate instrument and boundary.

The one-shot transformations do not get fine-tuning in the core comparison. The small students do get training. Therefore the table compares resulting artifacts under explicitly different construction procedures, not an equal-total-training-compute competition. A focused assignment extension should isolate one factor more tightly.

## 4. Worked decision reasoning

Suppose a factorized model has fewer parameters but slightly worse median latency, with overlapping timing variation. A valid conclusion is that storage/parameter savings were observed while a latency benefit was not established on this CPU workload. It is not valid to remove the slower measurement because theory predicted a speedup. Conversely, a small latency gain with a large quality loss may violate the chosen quality constraint.

Choose the decision rule before final test evaluation. For example, require accuracy within a predeclared tolerance of a reference and then minimize measured median latency, while checking p95. The tolerance is a project design choice, not a universal acceptable loss. Use validation to select compression settings and report the frozen final test comparison.

## 5. Error analysis and reproducibility

Compare predictions to identify which examples change after compression. Report whether the changes concentrate in particular classes. Inspect calibration if probabilities are part of the use case. Save factor ranks, masks, scales, student architecture and training configuration so a peer can reconstruct the artifact rather than merely load an unexplained binary file.

Retain per-run metrics and raw timing samples. A timing distribution from 100 repeated calls of one trained model is not the same as variation across 100 independently trained models. Label the source of uncertainty. Repeat complete benchmark sessions when close results could be explained by machine noise.

## Exercises and self-check

1. Compute frontiers for accuracy versus parameter bytes and accuracy versus latency. Explain why they can differ.
2. Include p95 latency and discuss whether a method attractive by median remains acceptable.
3. Explain which additional evidence would be needed to claim a phone-deployment speedup from the CPU experiment. **Check:** actual target hardware/backend/workload measurements.
4. Write a recommendation under a memory constraint and another under an interactive latency constraint. State when no tested model satisfies the requirement.

## Submission readiness

Submit a 3–5 page report, runnable artifacts, selection protocol, measured table, Pareto plot, raw timing samples and one controlled extension. Explain at least one case where theoretical savings did or did not translate into runtime. Correct measurement and clear limits matter more than obtaining a positive speedup.

## Primary reading

- [PyTorch: benchmark utilities](https://docs.pytorch.org/docs/2.9/benchmark_utils.html): audit the measurement procedure.
- [Deep Compression](https://arxiv.org/abs/1510.00149), [LoRA](https://arxiv.org/abs/2106.09685) and [Distillation](https://arxiv.org/abs/1503.02531): identify the exact mechanism corresponding to each artifact and avoid borrowing performance claims from different settings.
